In [ ]:
# Model imports
import pandas as pd
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

# Cross-validation imports
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import cross_val_score

# Data preprocessing imports
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [35]:
# Function to load dataset
def load_data(file_path):
    """
    Load dataset from a CSV file.
    
    Parameters:
    file_path (str): Path to the CSV file.
    
    Returns:
    pd.DataFrame: Loaded dataset.
    """
    data = pd.read_csv(file_path)
    print(f"Data loaded from {file_path} with shape {data.shape}")
    return data

data = load_data('../data/dataset.csv')
data.set_index("Game_ID", inplace=True)

prop_line = load_data('../data/avg_career_stats.csv')

data

Data loaded from ../data/dataset.csv with shape (86, 15)
Data loaded from ../data/avg_career_stats.csv with shape (1, 6)


,PTS,REB,AST,STL,BLK,FG3M,TRIPLE_DOUBLE,DOUBLE_DOUBLE,W,PLAYER_INACTIVE,PLAYER_DND,INACTIVE_ROLLING,DND_ROLLING,GAMES_MISSED_ROLLING
Game_ID,,,,,,,,,,,,,,
22401192,0,0,0,0,0,0,0,0,0,0,1,0,1,1
22401171,36,6,12,0,2,3,0,1,0,0,0,0,1,1
22401167,36,2,8,1,1,2,0,0,1,0,0,0,1,1
22401144,35,3,5,2,0,1,0,0,0,0,0,0,1,1
22401129,25,9,4,0,1,2,0,0,0,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22400063,28,5,8,1,0,2,0,0,0,0,0,0,0,0
22300857,21,3,10,1,0,2,0,1,0,0,0,0,0,0
22300675,19,2,7,2,1,2,0,0,0,0,0,0,0,0


In [ ]:
# Setting up K-Folds
rkf = RepeatedKFold(n_splits=10, n_repeats=10, random_state=101) 

pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())])

pipeline_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('model', XGBRegressor(n_estimators=200, learning_rate=0.01, max_depth=3, random_state=101))])

# Create a dataframe to hold the best model metrics for each stat
metrics = []

print("Training models and evaluating performance...")

for stat in ['PTS', 'REB', 'AST', 'FG3M', 'STL', 'BLK', 'TRIPLE_DOUBLE', 'DOUBLE_DOUBLE']:
    X_stat = data.drop(columns=[stat])
    y_stat = data[stat]
    
    # Linear Regression CV
    cv_results_lr = cross_val_score(pipeline_lr, X_stat, y_stat, cv=rkf, scoring='r2')
    mean_lr = cv_results_lr.mean()
    std_lr = cv_results_lr.std()
    
    # XGBoost CV
    cv_results_xgb = cross_val_score(pipeline_xgb, X_stat, y_stat, cv=rkf, scoring='r2')
    mean_xgb = cv_results_xgb.mean()
    std_xgb = cv_results_xgb.std()
    
    # Select best model
    if mean_lr > mean_xgb:
        best_model = 'Linear Regression'
        best_r2 = mean_lr
        best_std = std_lr
    else:
        best_model = 'XGBoost'
        best_r2 = mean_xgb
        best_std = std_xgb

    # If best R2 is negative, note in metrics and use mean for prediction
    if best_r2 < 0:
        metrics.append({
            'Stat': stat,
            'Model': 'Mean Prediction',
            'Mean R2': best_r2,
            'Std R2': best_std
        })
    else:
        metrics.append({
            'Stat': stat,
            'Model': best_model,
            'Mean R2': best_r2,
            'Std R2': best_std
        })

# Convert metrics list to DataFrame
metrics_df = pd.DataFrame(metrics)
print(metrics_df)



Training models and evaluating performance...
            Stat              Model   Mean R2    Std R2
0            PTS  Linear Regression  0.554176  0.463969
1            REB  Linear Regression  0.522706  0.322658
2            AST            XGBoost  0.689524  0.262989
3           FG3M  Linear Regression  0.301670  0.375483
4            STL    Mean Prediction -0.388997  0.873047
5            BLK    Mean Prediction -0.665449  1.256966
6  TRIPLE_DOUBLE            XGBoost  0.581427  0.474765
7  DOUBLE_DOUBLE            XGBoost  0.966865  0.097289
Predictions of player versus team:
{'PTS': 24.13633334582768, 'REB': 5.894465169020814, 'AST': 7.1920695, 'FG3M': 1.2122516661700793, 'STL': 0.8837209302325582, 'BLK': 0.627906976744186, 'TRIPLE_DOUBLE': 0.014388995, 'DOUBLE_DOUBLE': 0.0533704}


In [ ]:
# Predict each stat for the next game
predictions = {}
for stat in ['PTS', 'REB', 'AST', 'FG3M', 'STL', 'BLK', 'TRIPLE_DOUBLE', 'DOUBLE_DOUBLE']:
    pred_features = data.drop(columns=[stat])
    model_name = metrics_df[metrics_df['Stat'] == stat]['Model'].values[0]
    if model_name == 'Mean Prediction':
        predictions[stat] = data[stat].mean()
    elif model_name == 'Linear Regression':
        model = pipeline_lr.fit(data.drop(columns=[stat]), data[stat])
        predictions[stat] = model.predict(pred_features.tail(1))[0]
    else:
        model = pipeline_xgb.fit(data.drop(columns=[stat]), data[stat])
        predictions[stat] = model.predict(pred_features.tail(1))[0]

Predictions of player versus team:
{'PTS': 24.14, 'REB': 5.89, 'AST': 7.19, 'FG3M': 1.21, 'STL': 0.88, 'BLK': 0.63, 'TRIPLE_DOUBLE': 0.01, 'DOUBLE_DOUBLE': 0.05}


In [65]:
prop_line = prop_line.apply(lambda x: round(x * 2) / 2)

results = {}
for stat in predictions:
    if stat in prop_line.columns:
        prop_val = prop_line[stat].values[0]
        pred_val = predictions[stat]
        if pred_val < prop_val:
            results[stat] = f"UNDER ({pred_val:.2f}) vs Prop Line ({prop_val})"
        else:
            results[stat] = f"OVER ({pred_val:.2f}) vs Prop Line ({prop_val})"
    elif stat in ['TRIPLE_DOUBLE', 'DOUBLE_DOUBLE']:
        pred_val = predictions[stat]
        if pred_val < 0.5:
            results[stat] = f"UNDER ({pred_val:.2f})"
        else:
            results[stat] = f"OVER ({pred_val:.2f})"
results


{'PTS': 'UNDER (24.14) vs Prop Line (25.0)',
 'REB': 'UNDER (5.89) vs Prop Line (8.5)',
 'AST': 'UNDER (7.19) vs Prop Line (8.5)',
 'FG3M': 'OVER (1.21) vs Prop Line (1.0)',
 'STL': 'UNDER (0.88) vs Prop Line (2.0)',
 'BLK': 'UNDER (0.63) vs Prop Line (1.5)',
 'TRIPLE_DOUBLE': 'UNDER (0.01)',
 'DOUBLE_DOUBLE': 'UNDER (0.05)'}